In [ ]:
# Auto-Fix Working Directory 
import os
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
print("Working directory set to:", os.getcwd())


# Python for Biochemistry
## A Hands-on Workshop — Structural Bioinformatics with Biopython

**Author:** Omar Hamouda 

Environment:** Python 3.x | Jupyter Notebook | Anaconda

---

## 📌 Overview

This notebook covers the fundamentals of **structural bioinformatics** applied to real protein structures. Using **Biopython**, we will parse **mmCIF files** (macromolecular Crystallographic Information Files), explore their metadata, load 3D coordinates, and analyze **protein-ligand interactions** at the atomic level.

---

## 📚 Table of Contents

1. Setup & Imports
2. Reading MMCIF Metadata
3. Exploring the Dictionary
4. Loading 3D Structure
5. Finding the Iron Atom
6. Neighbor Search — Protein-Ligand Interactions
7. Challenge — Zinc Finger Analysis

---

## 📁 Data Files

- `1mbn.cif` — Myoglobin (Sperm Whale), X-ray, 2.00 Å
- `1a1t.cif` — HIV-1 Nucleocapsid Protein, NMR, 25 models

## ✏️ Test: Is the Notebook Working?

Quick sanity check: Verify that Biopython is installed and the kernel is running ✅

In [ ]:
# Verify Biopython installation
import Bio
print("Biopython version:", Bio.__version__)

---

## 1️⃣ Setup & Imports

### 🎯 Why?

Before any analysis, we need to:
- **Import** the required libraries.
- **Verify** that our data files exist.

> **Golden Rule:** Always verify files exist **before** reading them.

In [ ]:
# ─── Standard Library ───
import os

# ─── Biopython ───
from Bio.PDB.MMCIF2Dict import MMCIF2Dict      # Metadata → Dictionary
from Bio.PDB.MMCIFParser import MMCIFParser    # 3D Structure → Object
from Bio.PDB import NeighborSearch             # Spatial search

In [ ]:
# ─── Verify Data Files Exist ───
pdb_file_1mbn = os.path.join('data', 'PDB_files', '1mbn.cif')
pdb_file_1a1t = os.path.join('data', 'PDB_files', '1a1t.cif')

for f in [pdb_file_1mbn, pdb_file_1a1t]:
    if os.path.exists(f):
        print(f"✅ Found: {f}")
    else:
        print(f"❌ Missing: {f}")

---

## 2️⃣ Reading MMCIF Metadata

### 🎯 Why?

**mmCIF** (macromolecular Crystallographic Information File) is the modern standard format for storing protein structures. It contains **hundreds of data items** organized as `_category.attribute` pairs.

Instead of reading the file line-by-line, Biopython's `MMCIF2Dict` parses the entire file into a **Python dictionary** — allowing direct query by keyword.

### 💡 Key Concept

 category.attribute value
 
───────────────────

 cell.length_a 64.500
 
 entity.type polymer




> **Note:** Every value is returned as a **list**, because a keyword may hold multiple values (e.g., NMR models, multiple chains).

In [ ]:
# Read the CIF file into a dictionary
pdb_info = MMCIF2Dict("data/PDB_files/1mbn.cif")

# Inspect the type
print("Type:", type(pdb_info))

In [ ]:
# How many data items does this file contain?
print("Number of keys:", len(pdb_info))

In [ ]:
# Access cell parameters using dictionary syntax
print("Entry ID:   ", pdb_info["_cell.entry_id"])
print("Length a:   ", pdb_info["_cell.length_a"])
print("Length b:   ", pdb_info["_cell.length_b"])
print("Length c:   ", pdb_info["_cell.length_c"])

In [ ]:
# Extract more metadata: angles and Z (formula units per cell)
print("Alpha angle:      ", pdb_info["_cell.angle_alpha"])
print("Beta angle:       ", pdb_info["_cell.angle_beta"])
print("Gamma angle:      ", pdb_info["_cell.angle_gamma"])
print("Z (formula units):", pdb_info["_cell.Z_PDB"])

### 📝 Section Summary

In this section, we:
- **Loaded** an mmCIF file into a Python dictionary using `MMCIF2Dict`.
- **Verified** the file contains 400+ data items.
- **Extracted** cell parameters using dictionary syntax.

> **Key takeaway:** `MMCIF2Dict` gives us direct access to any data item via `_category.attribute` keys.

---

## 3️⃣ Exploring the Dictionary

### 🎯 Why?

An mmCIF file contains **hundreds of keys**. We need to **explore** them to find the data we're interested in.

Instead of dumping the entire dictionary (which is overwhelming), we **filter** by category:

- `_cell.` — Unit cell parameters
- `_entity.` — Description of each molecule (protein, ligand, water)
- `_entity_src_gen.` — Source organism
- `_entity_poly.` — Polymer sequence

> **Tip:** Use `"_keyword." in key` to filter by category.

In [ ]:
# Loop over all keys and print only "_cell." keys
for key, value in pdb_info.items():
    if "_cell." in key:
        print(key, "→", value)

In [ ]:
# Explore "_entity." keys — describes each molecule in the structure
for key, value in pdb_info.items():
    if "_entity." in key:
        print(key, "→", value)

In [ ]:
# Same loop, but filter out "?" (unknown values)
for key, value in pdb_info.items():
    if "?" not in value:
        if "_entity." in key:
            print(key, "→", value)

In [ ]:
# Extract organism information
common_name = pdb_info["_entity_src_gen.gene_src_common_name"]
scientific_name = pdb_info["_entity_src_gen.pdbx_gene_src_scientific_name"]

print("Common name:     ", common_name)
print("Scientific name: ", scientific_name)

In [ ]:
# Extract the protein sequence (one-letter code)
sequence = pdb_info["_entity_poly.pdbx_seq_one_letter_code"]
print("Sequence length:", len(sequence[0]))
print("\nFirst 80 amino acids:")
print(sequence[0][:80])

### 📝 Section Summary

In this section, we:
- **Filtered** the dictionary by category keyword (`"_cell."`, `"_entity."`).
- **Cleaned** the output by removing unknown values (`"?"`).
- **Extracted** the source organism (common + scientific name).
- **Retrieved** the protein sequence (154 amino acids).

> **Key takeaway:** Filtering the dictionary by `"_category."` is the fastest way to explore an mmCIF file.

---

## 4️⃣ Loading 3D Structure

### 🎯 Why?

Until now, `MMCIF2Dict` gave us **metadata** (text, numbers, descriptions). But we also need the **3D coordinates** of every atom to analyze interactions.

That's where `MMCIFParser` comes in — it loads the full structure into a **Structure Object** that follows the **SMCRA** hierarchy:


Structure

└ ── Model (1 for X-ray, many for NMR)

└ ── Chain (A, B, C, ...)

└ ── Residue (HIS 93, HEM 155, ...)

└ ── Atom (N, CA, C, O, FE, ...)




> **Key idea:** SMCRA gives us a **navigable tree** of the entire protein.

In [ ]:
# Create the parser (QUIET=True suppresses warnings)
parser = MMCIFParser(QUIET=True)

# Load the structure into a Biopython Structure object
structure = parser.get_structure("myoglobin", "data/PDB_files/1mbn.cif")

# Inspect the type
print("Type:", type(structure))
print("Structure ID:", structure.get_id())

In [ ]:
# Explore the SMCRA hierarchy
for model in structure:
    print(f"Model: {model.id}")
    for chain in model:
        print(f"  Chain: {chain.id}")
        for residue in chain:
            print(f"    Residue: {residue.get_resname()} {residue.get_id()[1]}")
            break  # فقط أول residue عشان الـ output يكون مختصر
        break      # فقط أول chain
    break          # فقط أول model

In [ ]:
# Count models, chains, residues, and atoms
num_models = len(list(structure.get_models()))
num_chains = len(list(structure.get_chains()))
num_residues = len(list(structure.get_residues()))
num_atoms = len(list(structure.get_atoms()))

print(f"Number of models:   {num_models}")
print(f"Number of chains:   {num_chains}")
print(f"Number of residues: {num_residues}")
print(f"Number of atoms:    {num_atoms}")

In [ ]:
# List all chains and their first residue
for chain in structure.get_chains():
    print(f"Chain {chain.id}:")
    residues = list(chain.get_residues())
    print(f"  Total residues: {len(residues)}")
    print(f"  First residue:  {residues[0].get_resname()} {residues[0].get_id()[1]}")
    print(f"  Last residue:   {residues[-1].get_resname()} {residues[-1].get_id()[1]}")

In [ ]:
# Get all atoms as a list (we'll need this for NeighborSearch later)
atom_list = list(structure.get_atoms())

# Inspect the first atom
first_atom = atom_list[0]
print("First atom:")
print(f"  Element: {first_atom.element}")
print(f"  Name:    {first_atom.get_name()}")
print(f"  Coord:   {first_atom.get_coord()}")
print(f"  Parent:  {first_atom.get_parent().get_resname()} {first_atom.get_parent().get_id()[1]}")

### 📝 Section Summary

In this section, we:
- **Loaded** an mmCIF file into a Biopython **Structure object**.
- **Explored** the SMCRA hierarchy (Model → Chain → Residue → Atom).
- **Counted** models, chains, residues, and atoms.
- **Inspected** the first atom's element, name, coordinates, and parent residue.

> **Key takeaway:** Every atom knows its parent residue via `get_parent()`. This is the foundation for interaction analysis.### 📝 Section Summary



---

## 5️⃣ Finding the Iron Atom

### 🎯 Why?

Myoglobin's function depends on the **iron atom (Fe)** in the heme group. Before analyzing interactions, we must **locate the Fe atom** in the structure.

**Strategy:**
1. Loop over all atoms.
2. Check if `element == "FE"`.
3. Save its coordinates.

> **Note:** Element symbols are **UPPERCASE** in Biopython (`"FE"`, not `"Fe"`).

In [ ]:
# Search for the iron atom
iron_atom = None

for atom in structure.get_atoms():
    if atom.element == "FE":
        iron_atom = atom
        break

# Report
print("Iron atom found:")
print(f"  Atom name:   {iron_atom.get_name()}")
print(f"  Coordinates: {iron_atom.get_coord()}")
print(f"  Parent:      {iron_atom.get_parent().get_resname()} {iron_atom.get_parent().get_id()[1]}")

In [ ]:
# Sanity check: inspect the iron atom object
print("Type:      ", type(iron_atom))
print("Element:   ", iron_atom.element)
print("B-factor:  ", iron_atom.get_bfactor())
print("Occupancy: ", iron_atom.get_occupancy())

In [ ]:
# Find ALL atoms of the heme group (HEM 155)
hem_atoms = []

for atom in structure.get_atoms():
    parent = atom.get_parent()
    if parent.get_resname() == "HEM":
        hem_atoms.append(atom)

print(f"Heme group has {len(hem_atoms)} atoms")
print("\nFirst 10 atoms:")
for atom in hem_atoms[:10]:
    print(f"  {atom.get_name():4s} ({atom.element})")

In [ ]:
# Extract iron coordinates for later use
iron_coord = iron_atom.get_coord()

print("Iron coordinates:", iron_coord)
print("Data type:       ", type(iron_coord))
print("X:               ", iron_coord[0])
print("Y:               ", iron_coord[1])
print("Z:               ", iron_coord[2])

### 📝 Section Summary

In this section, we:
- **Located** the iron atom by element (`"FE"`).
- **Verified** its identity (B-factor, occupancy).
- **Explored** the heme group (43 atoms).
- **Extracted** the iron's 3D coordinates.

> **Key takeaway:** Searching by `atom.element` is the fastest way to find specific atoms in a structure.

---

## 6️⃣ Neighbor Search — Protein-Ligand Interactions

### 🎯 Why?

Now that we have the iron coordinates, we can ask the scientific question:

> **Which protein residues bind to the heme iron?**

To answer this, we use **`NeighborSearch`** — a Biopython tool that finds all atoms within a **cutoff distance** of a target atom.

### 💡 Cutoff Distances (Rule of Thumb)

| Distance | Interaction Type |
|----------|------------------|
| 2–3 Å    | Strong bonds (H-bonds, coordination) |
| 3–4 Å    | Close interactions |
| **4–5 Å** | **Protein-ligand interactions** ⭐ |
| 6–8 Å    | Extended range |

> **Note:** `NeighborSearch` requires a **list** of atoms (not a generator).

In [ ]:
# Get all atoms as a list (required by NeighborSearch)
atom_list = list(structure.get_atoms())
print(f"Total atoms loaded: {len(atom_list)}")

# Create the NeighborSearch object
neighbor_search = NeighborSearch(atom_list)

print("NeighborSearch object created ✅")

In [ ]:
# Search for all atoms within 4 Å of the iron
cutoff_distance = 4  # Angstroms

neighbors = neighbor_search.search(iron_atom.get_coord(), cutoff_distance)

print(f"Cutoff distance: {cutoff_distance} Å")
print(f"Number of neighbors found: {len(neighbors)}")

In [ ]:
# Inspect the first 10 neighbors
print("First 10 neighbors of the iron atom:\n")
for neighbor in neighbors[:10]:
    residue = neighbor.get_parent()
    print(f"  {neighbor.element:3s} — {residue.get_resname()} {residue.get_id()[1]}")

In [ ]:
# Filter out atoms that belong to the SAME residue as the iron
# We only want PROTEIN atoms interacting with the ligand
iron_residue = iron_atom.get_parent()
print(f"Iron is in residue: {iron_residue.get_resname()} {iron_residue.get_id()[1]}\n")

protein_neighbors = []

for neighbor in neighbors:
    residue = neighbor.get_parent()
    if residue != iron_residue:   # ← exclude the heme itself
        protein_neighbors.append(neighbor)

print(f"Protein neighbors (excluding heme): {len(protein_neighbors)}\n")

for neighbor in protein_neighbors:
    residue = neighbor.get_parent()
    print(f"  {neighbor.element:3s} — {residue.get_resname()} {residue.get_id()[1]}")

In [ ]:
# Group neighbors by residue (some residues appear multiple times)
unique_residues = set()

for neighbor in protein_neighbors:
    residue = neighbor.get_parent()
    unique_residues.add((residue.get_resname(), residue.get_id()[1]))

print("Unique protein residues binding to the iron:\n")
for resname, resnum in sorted(unique_residues):
    print(f"  {resname} {resnum}")

### 🔬 Scientific Interpretation

Our analysis reveals the **key interactions** in myoglobin:

| Protein Residue | Atom | Distance to Fe |
|-----------------|------|----------------|
| **HIS 93**      | N    | ~2.0 Å         |
| **OH 154**      | O    | ~2.0 Å         |

**HIS 93** is known as the **"distal histidine"** — it stabilizes the bound oxygen molecule in the heme pocket.

**OH 154** (hydroxide ion) is directly coordinated to the iron in this structure (met-myoglobin form).

> **Biological insight:** The heme iron is coordinated by HIS 93 from the protein side, while the other side (distal) is available for oxygen binding. This is how myoglobin transports O₂ in muscle tissue.

### 📝 Section Summary

In this section, we:
- **Created** a `NeighborSearch` object from all atoms.
- **Searched** for atoms within 4 Å of the iron.
- **Filtered** out heme atoms to find only protein neighbors.
- **Grouped** neighbors by residue using a `set`.
- **Identified** HIS 93 and OH 154 as the key binding partners.

> **Key takeaway:** Neighbor search is the fundamental tool for analyzing protein-ligand interactions. The same workflow applies to any metal, ligand, or binding site.

---

## 7️⃣ Challenge — Zinc Finger Analysis

### 🎯 The Challenge

**Zinc fingers** are small protein motifs that coordinate **zinc ions (Zn²⁺)** to stabilize their structure. They are commonly found in **transcription factors** that bind DNA/RNA.

**Your mission:** Analyze the HIV-1 Nucleocapsid Protein (`1a1t.cif`) and answer:

1. **How many zinc atoms** does it contain?
2. **Which protein residues** coordinate each zinc?
3. **Why are NMR structures special** (compared to X-ray)?

### 💡 Hints

- **Hint 1:** `1a1t` is an **NMR structure** with **25 models**. Load only the **first model**.
- **Hint 2:** There are **2 zinc atoms** — analyze **both**.
- **Hint 3:** Zinc coordination typically involves **CYS** and **HIS** residues.

> **Note:** Zinc's element symbol in Biopython is `"ZN"` (uppercase).

In [ ]:
# Parse the 1a1t.cif structure (HIV-1 Nucleocapsid Protein)
parser_1a1t = MMCIFParser(QUIET=True)
structures_1a1t = parser_1a1t.get_structure("1a1t", "data/PDB_files/1a1t.cif")

# Check number of models (NMR)
num_models = len(list(structures_1a1t.get_models()))
print(f"Total models (NMR): {num_models}")

# Take only the first model
structure_1a1t = structures_1a1t[0]
print(f"Model used: {structure_1a1t.id}")

In [ ]:
# Find all zinc atoms in the structure
zinc_atoms = []

for atom in structure_1a1t.get_atoms():
    if atom.element == "ZN":
        zinc_atoms.append(atom)

print(f"Number of zinc atoms found: {len(zinc_atoms)}\n")

for i, zinc in enumerate(zinc_atoms, start=1):
    print(f"Zinc {i}:")
    print(f"  Coordinates: {zinc.get_coord()}")
    print(f"  Parent:      {zinc.get_parent().get_resname()} {zinc.get_parent().get_id()[1]}")

In [ ]:
# Create a NeighborSearch for the 1a1t structure
atom_list_1a1t = list(structure_1a1t.get_atoms())
neighbor_search_1a1t = NeighborSearch(atom_list_1a1t)

# Analyze each zinc atom separately
cutoff = 3  # Å — tighter cutoff for metal coordination

for i, zinc in enumerate(zinc_atoms, start=1):
    print(f"\n{'='*50}")
    print(f"  ZINC {i} — Neighbors within {cutoff} Å")
    print(f"{'='*50}")
    
    neighbors = neighbor_search_1a1t.search(zinc.get_coord(), cutoff)
    zinc_residue = zinc.get_parent()
    
    # Filter out atoms from the same zinc residue
    protein_neighbors = []
    for n in neighbors:
        n_residue = n.get_parent()
        if n_residue != zinc_residue:
            protein_neighbors.append(n)
    
    # Group by residue
    unique_residues = set()
    for n in protein_neighbors:
        res = n.get_parent()
        unique_residues.add((res.get_resname(), res.get_id()[1]))
    
    print(f"  Total neighbors:   {len(neighbors)}")
    print(f"  Protein neighbors: {len(protein_neighbors)}")
    print(f"  Coordinating residues:")
    for resname, resnum in sorted(unique_residues):
        print(f"    - {resname} {resnum}")

### 🔬 Scientific Interpretation

Our analysis reveals the zinc coordination pattern in the HIV-1 Nucleocapsid Protein:

| Zinc | Coordinating Residues | Motif Type |
|------|-----------------------|------------|
| Zn 1 | 3× CYS + 1× HIS       | Cys₃His    |
| Zn 2 | 3× CYS + 1× HIS       | Cys₃His    |

**Biological function:**
- This protein has **two zinc fingers** that grip the **viral RNA** during budding.
- The zinc ions are essential for structural stability.
- Without zinc, the protein loses its fold and cannot package viral RNA.

> **Medical relevance:** This motif is a **drug target** for HIV therapy — disrupting zinc coordination would prevent viral assembly.

### 📝 Section Summary

In this section, we:
- **Loaded** an NMR structure (25 models) and used only model 0.
- **Found** 2 zinc atoms using `element == "ZN"`.
- **Analyzed** each zinc's coordination sphere.
- **Identified** 3× CYS + 1× HIS per zinc (Cys₃His motif).
- **Interpreted** the medical relevance (HIV drug target).

> **Key takeaway:** The same Neighbor Search workflow applies to **any** metal or ligand — the only change is the element symbol and the cutoff distance.